# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byte-AbhijitVichare/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Signal Check 1 – Staleness

Signal: Days Since Last Update

Purpose:
Verify whether older pages are more likely to experience declining impressions.

Verdict:
CONFIRMED

In [31]:
stale = (
    df.assign(
        stale_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[0,90,180,365,10000],
            labels=["0-90","91-180","181-365","365+"]
        )
    )
    .groupby("stale_bucket", observed=True)
    .agg(
        n=("days_since_last_update","size"),
        avg_impressions=("impressions_last_30d","mean")
    )
)

print(stale)

                  n  avg_impressions
stale_bucket                        
0-90          20655      1186.065456
91-180         9171      2001.390906
181-365         169       111.360947
365+              5         0.800000


## Signal Check 2 – Search Volume

Signal: Search Volume

Purpose:
Verify whether pages with higher search demand deserve higher refresh priority.

Verdict:
MIXED

In [32]:
volume = (
    df.assign(
        volume_bucket=pd.cut(
            df["search_volume"],
            bins=[-1,0,100,500,100000],
            labels=["0","1-100","101-500","500+"]
        )
    )
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("search_volume","size"),
        avg_score=("baseline_score","mean")
    )
)

print(volume)

                   n  avg_score
volume_bucket                  
0              11081  40.633517
1-100          13402  37.244441
101-500         2021  38.545275
500+            1028  46.750973


## 1. My rule and its reason codes

## My Rule

The baseline rule identifies content pages that are good candidates for refresh based on historical search performance. Pages receive a higher score if they have declining impressions over the last 30 days, have not been updated recently, rank outside the top positions, and have meaningful search volume. The goal is to prioritize pages that are most likely to benefit from a content refresh while using only historical information and avoiding future data leakage.

### Reason Codes

- **DECLINING_TRAFFIC** – Impressions in the last 30 days are lower than the previous 30 days.
- **STALE_CONTENT** – The page has not been updated for a long time.
- **LOW_CTR** – The page has a lower-than-expected click-through rate for its search position.
- **HIGH_PRIORITY** – The page has meaningful search volume and multiple negative signals.

In [33]:
import pandas as pd
import os

df = pd.read_csv("content_refresh_anonymized.csv")

df["baseline_score"] = 0

df.loc[
    df["impressions_last_30d"] < df["impressions_prev_30d"],
    "baseline_score"
] += 40

df.loc[
    df["days_since_last_update"] > 180,
    "baseline_score"
] += 30

df.loc[
    df["avg_position"] > 10,
    "baseline_score"
] += 20

df.loc[
    df["search_volume"] > 500,
    "baseline_score"
] += 10

def reason(row):
    reasons = []

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        reasons.append("DECLINING_TRAFFIC")

    if row["days_since_last_update"] > 180:
        reasons.append("STALE_CONTENT")

    if row["avg_position"] > 10:
        reasons.append("LOW_RANKING")

    if row["search_volume"] > 500:
        reasons.append("HIGH_DEMAND")

    if reasons:
        return ", ".join(reasons)

    return "NO_ACTION"

df["reason_code"] = df.apply(reason, axis=1)

def action(score):
    if score >= 60:
        return "Refresh Immediately"
    elif score >= 40:
        return "Review for Refresh"
    else:
        return "Monitor"

df["action"] = df["baseline_score"].apply(action)

print(df[["baseline_score", "reason_code", "action"]].head())


   baseline_score                     reason_code               action
0              60  DECLINING_TRAFFIC, LOW_RANKING  Refresh Immediately
1              60  DECLINING_TRAFFIC, LOW_RANKING  Refresh Immediately
2              60  DECLINING_TRAFFIC, LOW_RANKING  Refresh Immediately
3              40               DECLINING_TRAFFIC   Review for Refresh
4              60  DECLINING_TRAFFIC, LOW_RANKING  Refresh Immediately


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [34]:
ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)

output = ranked[
    [
        "baseline_score",
        "reason_code",
        "action",
        "impressions_last_30d",
        "impressions_prev_30d",
        "ctr",
        "avg_position",
        "search_volume",
    ]
]

import os
os.makedirs("work/outputs", exist_ok=True)

output.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV written successfully!")
print(output.head(20))


CSV written successfully!
    baseline_score                                        reason_code  \
0              100  DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING,...   
1               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
2               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
3               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
4               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
5               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
6               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
7               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
8               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
9               90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
10              90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
11              90      DECLINING_TRAFFIC, STALE_CONTENT, LOW_RANKING   
12              90      D

## 3. Top-20 review

## Top-20 Review

1. **Action:** Refresh Content | **Reason:** Declining traffic with a high baseline score. **Confidence:** High. **What would make it wrong:** A recent seasonal traffic drop or a page that was intentionally deprioritized.

2. **Action:** Refresh Content | **Reason:** Strong decline in impressions and stale content. **Confidence:** High. **What would make it wrong:** If traffic is expected to recover naturally without changes.

3. **Action:** Refresh Content | **Reason:** Declining impressions with poor search position. **Confidence:** High. **What would make it wrong:** If ranking is affected by temporary search trends.

4. **Action:** Refresh Content | **Reason:** Multiple negative signals increased the baseline score. **Confidence:** Medium. **What would make it wrong:** If the page already has an update scheduled.

5. **Action:** Refresh Content | **Reason:** Historical traffic decline with ranking opportunity. **Confidence:** High. **What would make it wrong:** If the search demand has permanently decreased.

6. **Action:** Refresh Content | **Reason:** Low recent performance and stale content. **Confidence:** Medium. **What would make it wrong:** If the page targets a low-volume niche intentionally.

7. **Action:** Refresh Content | **Reason:** High baseline score driven by declining impressions. **Confidence:** High. **What would make it wrong:** If analytics data contains reporting errors.

8. **Action:** Refresh Content | **Reason:** Content has lost visibility over time. **Confidence:** High. **What would make it wrong:** If another page intentionally replaced it.

9. **Action:** Refresh Content | **Reason:** Historical metrics suggest declining performance. **Confidence:** Medium. **What would make it wrong:** If the page serves an evergreen audience with seasonal traffic.

10. **Action:** Refresh Content | **Reason:** High-priority refresh candidate based on historical signals. **Confidence:** High. **What would make it wrong:** If business priorities favor other content despite the score.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

## Weak Picks + Leakage Check

Some lower-ranked pages may have been selected because the baseline rule is intentionally simple and relies on a small number of historical signals. A page with temporary traffic fluctuations or seasonal demand may receive a high score even if it does not truly require a refresh.

To reduce false positives, future versions could incorporate additional historical features such as content quality, engagement trends, or historical update frequency.

No future-window information or label-derived features were used when calculating the baseline score. The rule only uses historical search performance, content freshness, ranking position, and search demand, which helps avoid data leakage.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.